In [1]:
# For Production
import json
import random
import requests
import numpy as np
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# Colab only
from google.colab import userdata

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## Data

#### My Anime List

In [3]:
mal_client_id = userdata.get('MyAnimeList')

headers = {
    'X-MAL-CLIENT-ID': mal_client_id
}


In [4]:
## Extraccion de anime tipo TV
mal_client_id = userdata.get('MyAnimeList')
headers = {'X-MAL-CLIENT-ID': mal_client_id}

todos = []
for offset in range(0, 500, 100):
    res = requests.get('https://api.myanimelist.net/v2/anime/ranking', params={
        'ranking_type': 'tv',
        'limit': 100,
        'offset': offset,
        'fields': 'rank,mean,genres,num_episodes,status,start_date,synopsis,popularity,media_type'
    }, headers=headers).json()

    for item in res.get('data', []):
        node = item['node']
        todos.append({
            'mal_id':     node['id'],
            'rank':       item['ranking']['rank'],
            'title':      node['title'],
            'score':      node.get('mean'),
            'episodes':   node.get('num_episodes'),
            'status':     node.get('status'),
            'media_type': node.get('media_type'),
            'genres':     ', '.join([g['name'] for g in node.get('genres', [])]),
            'year':       node.get('start_date', '')[:4],
            'popularity': node.get('popularity'),
            'synopsis':   node.get('synopsis'),
        })
    time.sleep(1)

tv = pd.DataFrame(todos)
print(f" {len(tv)} animes extraídos del ranking")
tv.head()

 500 animes extraídos del ranking


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis
0,52991,1,Sousou no Frieren,9.27,28,finished_airing,tv,"Adventure, Award Winning, Drama, Fantasy, Shounen",2023,104,During their decade-long quest to defeat the D...
1,5114,2,Fullmetal Alchemist: Brotherhood,9.11,64,finished_airing,tv,"Action, Adventure, Drama, Fantasy, Military, S...",2009,3,After a horrific alchemy experiment goes wrong...
2,9253,3,Steins;Gate,9.07,24,finished_airing,tv,"Drama, Psychological, Sci-Fi, Suspense, Time T...",2011,14,Eccentric scientist Rintarou Okabe has a never...
3,28977,4,Gintama°,9.05,51,finished_airing,tv,"Action, Comedy, Gag Humor, Historical, Parody,...",2015,349,"Gintoki, Shinpachi, and Kagura return as the f..."
4,38524,5,Shingeki no Kyojin Season 3 Part 2,9.05,10,finished_airing,tv,"Action, Drama, Gore, Military, Shounen, Surviv...",2019,20,Seeking to restore humanity's diminishing hope...


In [5]:
## Extracción de detalles para el funcionamiento del recomendador
DETAIL_FIELDS = 'studios,rating,main_picture,related_anime,themes,demographics'

def fetch_detail(row):
    for intento in range(3):
        try:
            res = requests.get(
                f'https://api.myanimelist.net/v2/anime/{row["mal_id"]}',
                params={'fields': DETAIL_FIELDS},
                headers=headers,
                timeout=10
            )
            if res.status_code == 200:
                data = res.json()
                return {
                    'mal_id'       : row['mal_id'],
                    'studios'      : ', '.join([s['name'] for s in data.get('studios', [])]),
                    'rating'       : data.get('rating'),
                    'image_url'    : data.get('main_picture', {}).get('large'),
                    'related_anime': [r['node']['title'] for r in data.get('related_anime', [])],
                }
            elif res.status_code == 429:
                time.sleep(2 ** intento)
        except Exception as e:
            time.sleep(1)

    return {'mal_id': row['mal_id']}

detalles = []
rows = [row for _, row in tv.iterrows()]

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(fetch_detail, row): row for row in rows}
    for i, future in enumerate(as_completed(futures)):
        detalles.append(future.result())
        if (i + 1) % 50 == 0:
            print(f"   {i + 1}/{len(rows)} procesados...")

df_detalles = pd.DataFrame(detalles)
tv = tv.merge(df_detalles, on='mal_id', how='left')

print(f"tv enriquecido: {tv.shape}")
tv.head(3)

   50/500 procesados...
   100/500 procesados...
   150/500 procesados...
   200/500 procesados...
   250/500 procesados...
   300/500 procesados...
   350/500 procesados...
   400/500 procesados...
   450/500 procesados...
   500/500 procesados...
tv enriquecido: (500, 15)


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis,studios,rating,image_url,related_anime
0,52991,1,Sousou no Frieren,9.27,28,finished_airing,tv,"Adventure, Award Winning, Drama, Fantasy, Shounen",2023,104,During their decade-long quest to defeat the D...,Madhouse,pg_13,https://myanimelist.net/images/anime/1015/1380...,"[Yuusha, Sousou no Frieren: ●● no Mahou, Haru ..."
1,5114,2,Fullmetal Alchemist: Brotherhood,9.11,64,finished_airing,tv,"Action, Adventure, Drama, Fantasy, Military, S...",2009,3,After a horrific alchemy experiment goes wrong...,Bones,r,https://myanimelist.net/images/anime/1208/9474...,"[Fullmetal Alchemist, Fullmetal Alchemist: Bro..."
2,9253,3,Steins;Gate,9.07,24,finished_airing,tv,"Drama, Psychological, Sci-Fi, Suspense, Time T...",2011,14,Eccentric scientist Rintarou Okabe has a never...,White Fox,pg_13,https://myanimelist.net/images/anime/1935/1279...,"[ChäoS;HEAd, Steins;Gate: Oukoubakko no Poriom..."


In [6]:
## Extraccion de anime tipo Movie
mal_client_id = userdata.get('MyAnimeList')
headers = {'X-MAL-CLIENT-ID': mal_client_id}

todos = []
for offset in range(0, 500, 100):
    res = requests.get('https://api.myanimelist.net/v2/anime/ranking', params={
        'ranking_type': 'movie',
        'limit': 100,
        'offset': offset,
        'fields': 'rank,mean,genres,num_episodes,status,start_date,synopsis,popularity,media_type'
    }, headers=headers).json()

    for item in res.get('data', []):
        node = item['node']
        todos.append({
            'mal_id':     node['id'],
            'rank':       item['ranking']['rank'],
            'title':      node['title'],
            'score':      node.get('mean'),
            'episodes':   node.get('num_episodes'),
            'status':     node.get('status'),
            'media_type': node.get('media_type'),
            'genres':     ', '.join([g['name'] for g in node.get('genres', [])]),
            'year':       node.get('start_date', '')[:4],
            'popularity': node.get('popularity'),
            'synopsis':   node.get('synopsis'),
        })
    time.sleep(1)

movie = pd.DataFrame(todos)
print(f" {len(movie)} animes extraídos del ranking")
movie.head()

 500 animes extraídos del ranking


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis
0,57555,1,Chainsaw Man Movie: Reze-hen,9.09,1,finished_airing,movie,"Action, Fantasy, Gore, Shounen, Urban Fantasy",2025,591,Despite the immediate challenges following bec...
1,39486,2,Gintama: The Final,9.05,1,finished_airing,movie,"Action, Comedy, Drama, Gag Humor, Historical, ...",2021,1503,Two years have passed following the Tendoshuu'...
2,28851,3,Koe no Katachi,8.93,1,finished_airing,movie,"Award Winning, Drama, Shounen",2016,19,"As a wild youth, elementary school student Sho..."
3,15335,4,Gintama Movie 2: Kanketsu-hen - Yorozuya yo Ei...,8.89,1,finished_airing,movie,"Action, Comedy, Gag Humor, Historical, Parody,...",2013,1071,When Gintoki apprehends a movie pirate at a pr...
4,59571,5,Shingeki no Kyojin Movie: Kanketsu-hen - The L...,8.83,1,finished_airing,movie,"Action, Drama, Gore, Military, Shounen, Surviv...",2024,2644,A compilation movie for Shingeki no Kyojin: Th...


In [7]:
## Extracción de detalles para el funcionamiento del recomendador
DETAIL_FIELDS = 'studios,rating,main_picture,related_anime,themes,demographics'

def fetch_detail(row):
    for intento in range(3):
        try:
            res = requests.get(
                f'https://api.myanimelist.net/v2/anime/{row["mal_id"]}',
                params={'fields': DETAIL_FIELDS},
                headers=headers,
                timeout=10
            )
            if res.status_code == 200:
                data = res.json()
                return {
                    'mal_id'       : row['mal_id'],
                    'studios'      : ', '.join([s['name'] for s in data.get('studios', [])]),
                    'rating'       : data.get('rating'),
                    'image_url'    : data.get('main_picture', {}).get('large'),
                    'related_anime': [r['node']['title'] for r in data.get('related_anime', [])],
                }
            elif res.status_code == 429:
                time.sleep(2 ** intento)
        except Exception as e:
            time.sleep(1)

    return {'mal_id': row['mal_id']}

detalles = []
rows = [row for _, row in movie.iterrows()]

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(fetch_detail, row): row for row in rows}
    for i, future in enumerate(as_completed(futures)):
        detalles.append(future.result())
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(rows)} procesados...")

df_detalles = pd.DataFrame(detalles)
movie = movie.merge(df_detalles, on='mal_id', how='left')

print(f" movie enriquecido: {movie.shape}")
movie.head(3)

  50/500 procesados...
  100/500 procesados...
  150/500 procesados...
  200/500 procesados...
  250/500 procesados...
  300/500 procesados...
  350/500 procesados...
  400/500 procesados...
  450/500 procesados...
  500/500 procesados...
 movie enriquecido: (500, 15)


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis,studios,rating,image_url,related_anime
0,57555,1,Chainsaw Man Movie: Reze-hen,9.09,1,finished_airing,movie,"Action, Fantasy, Gore, Shounen, Urban Fantasy",2025,591,Despite the immediate challenges following bec...,MAPPA,r,https://myanimelist.net/images/anime/1763/1506...,"[Chainsaw Man, Chainsaw Man: Shikaku-hen]"
1,39486,2,Gintama: The Final,9.05,1,finished_airing,movie,"Action, Comedy, Drama, Gag Humor, Historical, ...",2021,1503,Two years have passed following the Tendoshuu'...,Bandai Namco Pictures,pg_13,https://myanimelist.net/images/anime/1245/1167...,"[Gintama: The Semi-Final, Gintama: The Final x..."
2,28851,3,Koe no Katachi,8.93,1,finished_airing,movie,"Award Winning, Drama, Shounen",2016,19,"As a wild youth, elementary school student Sho...",Kyoto Animation,pg_13,https://myanimelist.net/images/anime/1122/9643...,[Koe no Katachi Specials]


In [9]:
## Extraccion de anime tipo OVA
mal_client_id = userdata.get('MyAnimeList')
headers = {'X-MAL-CLIENT-ID': mal_client_id}

todos = []
for offset in range(0, 500, 100):
    res = requests.get('https://api.myanimelist.net/v2/anime/ranking', params={
        'ranking_type': 'ova',
        'limit': 100,
        'offset': offset,
        'fields': 'rank,mean,genres,num_episodes,status,start_date,synopsis,popularity,media_type'
    }, headers=headers).json()

    for item in res.get('data', []):
        node = item['node']
        todos.append({
            'mal_id':     node['id'],
            'rank':       item['ranking']['rank'],
            'title':      node['title'],
            'score':      node.get('mean'),
            'episodes':   node.get('num_episodes'),
            'status':     node.get('status'),
            'media_type': node.get('media_type'),
            'genres':     ', '.join([g['name'] for g in node.get('genres', [])]),
            'year':       node.get('start_date', '')[:4],
            'popularity': node.get('popularity'),
            'synopsis':   node.get('synopsis'),
        })
    time.sleep(1)

ova = pd.DataFrame(todos)
print(f" {len(ova)} animes extraídos del ranking")
ova.head()

 500 animes extraídos del ranking


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis
0,820,1,Ginga Eiyuu Densetsu,9.02,110,finished_airing,ova,"Adult Cast, Drama, Military, Sci-Fi, Space",1988,755,The 150-year-long stalemate between the two in...
1,44,2,Rurouni Kenshin: Meiji Kenkaku Romantan - Tsui...,8.69,4,finished_airing,ova,"Action, Adult Cast, Drama, Historical, Romance...",1999,950,When mankind's savagery surpasses his fear of ...
2,25781,3,Shingeki no Kyojin: Kuinaki Sentaku,8.42,2,finished_airing,ova,Action,2014,440,Many years before becoming the famed captain o...
3,30709,4,Kamisama Hajimemashita: Kako-hen,8.41,4,finished_airing,ova,"Comedy, Mythology, Romance, Shoujo, Supernatur...",2015,1487,While playing in the snow one day at her shrin...
4,32366,5,Gintama°: Aizome Kaori-hen,8.38,2,finished_airing,ova,"Comedy, Gag Humor, Parody, Shounen",2016,2502,"The red-light district, Yoshiwara, is suddenly..."


In [10]:
## Extracción de detalles para el funcionamiento del recomendador
DETAIL_FIELDS = 'studios,rating,main_picture,related_anime,themes,demographics'

def fetch_detail(row):
    for intento in range(3):
        try:
            res = requests.get(
                f'https://api.myanimelist.net/v2/anime/{row["mal_id"]}',
                params={'fields': DETAIL_FIELDS},
                headers=headers,
                timeout=10
            )
            if res.status_code == 200:
                data = res.json()
                return {
                    'mal_id'       : row['mal_id'],
                    'studios'      : ', '.join([s['name'] for s in data.get('studios', [])]),
                    'rating'       : data.get('rating'),
                    'image_url'    : data.get('main_picture', {}).get('large'),
                    'related_anime': [r['node']['title'] for r in data.get('related_anime', [])],
                }
            elif res.status_code == 429:
                time.sleep(2 ** intento)
        except Exception as e:
            time.sleep(1)

    return {'mal_id': row['mal_id']}

detalles = []
rows = [row for _, row in ova.iterrows()]

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(fetch_detail, row): row for row in rows}
    for i, future in enumerate(as_completed(futures)):
        detalles.append(future.result())
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(rows)} procesados...")

df_detalles = pd.DataFrame(detalles)
ova = ova.merge(df_detalles, on='mal_id', how='left')

print(f" Ova enriquecido: {ova.shape}")
ova.head(3)

  50/500 procesados...
  100/500 procesados...
  150/500 procesados...
  200/500 procesados...
  250/500 procesados...
  300/500 procesados...
  350/500 procesados...
  400/500 procesados...
  450/500 procesados...
  500/500 procesados...
 Ova enriquecido: (500, 15)


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis,studios,rating,image_url,related_anime
0,820,1,Ginga Eiyuu Densetsu,9.02,110,finished_airing,ova,"Adult Cast, Drama, Military, Sci-Fi, Space",1988,755,The 150-year-long stalemate between the two in...,"K-Factory, Kitty Film Mitaka Studio",r,https://myanimelist.net/images/anime/1976/1420...,[Ginga Eiyuu Densetsu: Arata naru Tatakai no O...
1,44,2,Rurouni Kenshin: Meiji Kenkaku Romantan - Tsui...,8.69,4,finished_airing,ova,"Action, Adult Cast, Drama, Historical, Romance...",1999,950,When mankind's savagery surpasses his fear of ...,Studio Deen,r,https://myanimelist.net/images/anime/1656/1376...,[Rurouni Kenshin: Meiji Kenkaku Romantan]
2,25781,3,Shingeki no Kyojin: Kuinaki Sentaku,8.42,2,finished_airing,ova,Action,2014,440,Many years before becoming the famed captain o...,Wit Studio,r,https://myanimelist.net/images/anime/8/69497l....,[Shingeki no Kyojin]


In [11]:
## Extraccion de anime tipo Special
mal_client_id = userdata.get('MyAnimeList')
headers = {'X-MAL-CLIENT-ID': mal_client_id}

todos = []
for offset in range(0, 500, 100):
    res = requests.get('https://api.myanimelist.net/v2/anime/ranking', params={
        'ranking_type': 'special',
        'limit': 100,
        'offset': offset,
        'fields': 'rank,mean,genres,num_episodes,status,start_date,synopsis,popularity,media_type'
    }, headers=headers).json()

    for item in res.get('data', []):
        node = item['node']
        todos.append({
            'mal_id':     node['id'],
            'rank':       item['ranking']['rank'],
            'title':      node['title'],
            'score':      node.get('mean'),
            'episodes':   node.get('num_episodes'),
            'status':     node.get('status'),
            'media_type': node.get('media_type'),
            'genres':     ', '.join([g['name'] for g in node.get('genres', [])]),
            'year':       node.get('start_date', '')[:4],
            'popularity': node.get('popularity'),
            'synopsis':   node.get('synopsis'),
        })
    time.sleep(1)

special = pd.DataFrame(todos)
print(f" {len(special)} animes extraídos del ranking")
special.head()

 500 animes extraídos del ranking


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis
0,60022,1,One Piece Fan Letter,9.02,1,finished_airing,tv_special,"Action, Adventure, Fantasy, Shounen",2024,1752,Although the golden age of piracy is about to ...
1,35247,2,Owarimonogatari 2nd Season,8.86,7,finished_airing,tv_special,"Comedy, Mystery, Supernatural, Vampire",2017,613,Following an encounter with oddity specialist ...
2,51535,3,Shingeki no Kyojin: The Final Season - Kankets...,8.86,2,finished_airing,tv_special,"Action, Drama, Gore, Military, Shounen, Surviv...",2023,250,In the wake of Eren Yeager's cataclysmic actio...
3,61903,4,Kaguya-sama wa Kokurasetai: Otona e no Kaidan,8.54,1,finished_airing,tv_special,"Comedy, Romance, School, Seinen",2025,2407,"Years after graduating from high school, Kaguy..."
4,21329,5,Mushishi: Hihamukage,8.53,1,finished_airing,tv_special,"Adult Cast, Adventure, Historical, Iyashikei, ...",2014,1735,The entire countryside comes to a halt midday ...


In [12]:
## Extracción de detalles para el funcionamiento del recomendador
DETAIL_FIELDS = 'studios,rating,main_picture,related_anime,themes,demographics'

def fetch_detail(row):
    for intento in range(3):
        try:
            res = requests.get(
                f'https://api.myanimelist.net/v2/anime/{row["mal_id"]}',
                params={'fields': DETAIL_FIELDS},
                headers=headers,
                timeout=10
            )
            if res.status_code == 200:
                data = res.json()
                return {
                    'mal_id'       : row['mal_id'],
                    'studios'      : ', '.join([s['name'] for s in data.get('studios', [])]),
                    'rating'       : data.get('rating'),
                    'image_url'    : data.get('main_picture', {}).get('large'),
                    'related_anime': [r['node']['title'] for r in data.get('related_anime', [])],
                }
            elif res.status_code == 429:
                time.sleep(2 ** intento)
        except Exception as e:
            time.sleep(1)

    return {'mal_id': row['mal_id']}

detalles = []
rows = [row for _, row in special.iterrows()]

with ThreadPoolExecutor(max_workers=4) as executor:  # ✅ subido a 4
    futures = {executor.submit(fetch_detail, row): row for row in rows}
    for i, future in enumerate(as_completed(futures)):
        detalles.append(future.result())
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(rows)} procesados...")

df_detalles = pd.DataFrame(detalles)
special = special.merge(df_detalles, on='mal_id', how='left')

print(f" special enriquecido: {special.shape}")
special.head(3)

  50/500 procesados...
  100/500 procesados...
  150/500 procesados...
  200/500 procesados...
  250/500 procesados...
  300/500 procesados...
  350/500 procesados...
  400/500 procesados...
  450/500 procesados...
  500/500 procesados...
 special enriquecido: (500, 15)


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis,studios,rating,image_url,related_anime
0,60022,1,One Piece Fan Letter,9.02,1,finished_airing,tv_special,"Action, Adventure, Fantasy, Shounen",2024,1752,Although the golden age of piracy is about to ...,NaN,NaN,NaN,NaN
1,35247,2,Owarimonogatari 2nd Season,8.86,7,finished_airing,tv_special,"Comedy, Mystery, Supernatural, Vampire",2017,613,Following an encounter with oddity specialist ...,NaN,NaN,NaN,NaN
2,51535,3,Shingeki no Kyojin: The Final Season - Kankets...,8.86,2,finished_airing,tv_special,"Action, Drama, Gore, Military, Shounen, Surviv...",2023,250,In the wake of Eren Yeager's cataclysmic actio...,NaN,NaN,NaN,NaN


In [14]:
## Extraccion de anime tipo Favorito
mal_client_id = userdata.get('MyAnimeList')
headers = {'X-MAL-CLIENT-ID': mal_client_id}

todos = []
for offset in range(0, 500, 100):
    res = requests.get('https://api.myanimelist.net/v2/anime/ranking', params={
        'ranking_type': 'favorite',
        'limit': 100,
        'offset': offset,
        'fields': 'rank,mean,genres,num_episodes,status,start_date,synopsis,popularity,media_type'
    }, headers=headers).json()

    for item in res.get('data', []):
        node = item['node']
        todos.append({
            'mal_id':     node['id'],
            'rank':       item['ranking']['rank'],
            'title':      node['title'],
            'score':      node.get('mean'),
            'episodes':   node.get('num_episodes'),
            'status':     node.get('status'),
            'media_type': node.get('media_type'),
            'genres':     ', '.join([g['name'] for g in node.get('genres', [])]),
            'year':       node.get('start_date', '')[:4],
            'popularity': node.get('popularity'),
            'synopsis':   node.get('synopsis'),
        })
    time.sleep(1)

favorite = pd.DataFrame(todos)
print(f"{len(favorite)} animes extraídos del ranking")
favorite.head()

500 animes extraídos del ranking


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis
0,21,1,One Piece,8.73,0,currently_airing,tv,"Action, Adventure, Fantasy, Shounen",1999,17,Barely surviving in a barrel after passing thr...
1,5114,2,Fullmetal Alchemist: Brotherhood,9.11,64,finished_airing,tv,"Action, Adventure, Drama, Fantasy, Military, S...",2009,3,After a horrific alchemy experiment goes wrong...
2,11061,3,Hunter x Hunter (2011),9.03,148,finished_airing,tv,"Action, Adventure, Fantasy, Shounen",2011,8,Hunters devote themselves to accomplishing haz...
3,9253,4,Steins;Gate,9.07,24,finished_airing,tv,"Drama, Psychological, Sci-Fi, Suspense, Time T...",2011,14,Eccentric scientist Rintarou Okabe has a never...
4,16498,5,Shingeki no Kyojin,8.57,25,finished_airing,tv,"Action, Award Winning, Drama, Gore, Military, ...",2013,1,"Centuries ago, mankind was slaughtered to near..."


In [15]:
## Extracción de detalles para el funcionamiento del recomendador
DETAIL_FIELDS = 'studios,rating,main_picture,related_anime,themes,demographics'

def fetch_detail(row):
    for intento in range(3):
        try:
            res = requests.get(
                f'https://api.myanimelist.net/v2/anime/{row["mal_id"]}',
                params={'fields': DETAIL_FIELDS},
                headers=headers,
                timeout=10
            )
            if res.status_code == 200:
                data = res.json()
                return {
                    'mal_id'       : row['mal_id'],
                    'studios'      : ', '.join([s['name'] for s in data.get('studios', [])]),
                    'rating'       : data.get('rating'),
                    'image_url'    : data.get('main_picture', {}).get('large'),
                    'related_anime': [r['node']['title'] for r in data.get('related_anime', [])],
                }
            elif res.status_code == 429:
                time.sleep(2 ** intento)
        except Exception as e:
            time.sleep(1)

    return {'mal_id': row['mal_id']}

detalles = []
rows = [row for _, row in favorite.iterrows()]

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(fetch_detail, row): row for row in rows}
    for i, future in enumerate(as_completed(futures)):
        detalles.append(future.result())
        if (i + 1) % 50 == 0:
            print(f"   {i + 1}/{len(rows)} procesados...")

df_detalles = pd.DataFrame(detalles)
favorite = favorite.merge(df_detalles, on='mal_id', how='left')

print(f" favorite enriquecido: {favorite.shape}")
favorite.head(3)

   50/500 procesados...
   100/500 procesados...
   150/500 procesados...
   200/500 procesados...
   250/500 procesados...
   300/500 procesados...
   350/500 procesados...
   400/500 procesados...
   450/500 procesados...
   500/500 procesados...
 favorite enriquecido: (500, 15)


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis,studios,rating,image_url,related_anime
0,21,1,One Piece,8.73,0,currently_airing,tv,"Action, Adventure, Fantasy, Shounen",1999,17,Barely surviving in a barrel after passing thr...,Toei Animation,pg_13,https://myanimelist.net/images/anime/1244/1388...,"[One Piece Movie 01, One Piece Movie 02: Nejim..."
1,5114,2,Fullmetal Alchemist: Brotherhood,9.11,64,finished_airing,tv,"Action, Adventure, Drama, Fantasy, Military, S...",2009,3,After a horrific alchemy experiment goes wrong...,Bones,r,https://myanimelist.net/images/anime/1208/9474...,"[Fullmetal Alchemist, Fullmetal Alchemist: Bro..."
2,11061,3,Hunter x Hunter (2011),9.03,148,finished_airing,tv,"Action, Adventure, Fantasy, Shounen",2011,8,Hunters devote themselves to accomplishing haz...,Madhouse,pg_13,https://myanimelist.net/images/anime/1337/9901...,"[Hunter x Hunter, Hunter x Hunter: Original Vi..."


In [16]:
## Extraccion de anime en emisión
mal_client_id = userdata.get('MyAnimeList')
headers = {'X-MAL-CLIENT-ID': mal_client_id}

todos = []
for offset in range(0, 500, 100):
    res = requests.get('https://api.myanimelist.net/v2/anime/ranking', params={
        'ranking_type': 'airing',
        'limit': 100,
        'offset': offset,
        'fields': 'rank,mean,genres,num_episodes,status,start_date,synopsis,popularity,media_type'
    }, headers=headers).json()

    for item in res.get('data', []):
        node = item['node']
        todos.append({
            'mal_id':     node['id'],
            'rank':       item['ranking']['rank'],
            'title':      node['title'],
            'score':      node.get('mean'),
            'episodes':   node.get('num_episodes'),
            'status':     node.get('status'),
            'media_type': node.get('media_type'),
            'genres':     ', '.join([g['name'] for g in node.get('genres', [])]),
            'year':       node.get('start_date', '')[:4],
            'popularity': node.get('popularity'),
            'synopsis':   node.get('synopsis'),
        })
    time.sleep(1)

airing = pd.DataFrame(todos)
print(f"{len(airing)} animes extraídos del ranking")
airing.head()

349 animes extraídos del ranking


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis
0,61469,1,Steel Ball Run: JoJo no Kimyou na Bouken,9.17,0,currently_airing,ona,"Action, Adventure, Historical, Mystery, Racing...",2026,1484,"In the American Old West, the world's greatest..."
1,51553,2,Tongari Boushi no Atelier,8.80,13,currently_airing,tv,"Fantasy, Seinen",2026,1824,"Coco, a humble dressmaker's daughter, has alwa..."
2,21,3,One Piece,8.73,0,currently_airing,tv,"Action, Adventure, Fantasy, Shounen",1999,17,Barely surviving in a barrel after passing thr...
3,50250,4,Chiikawa,8.60,0,currently_airing,tv,Slice of Life,2022,5524,"What's a Chiikawa? No one really knows, but ev..."
4,60988,5,Tian Guan Cifu Short Films,8.49,0,currently_airing,ona,"Action, Adventure, Drama, Fantasy, Historical,...",2025,6945,A series of short films.\n\nShort film #1: Xie...


In [17]:
## Extracción de detalles para el funcionamiento del recomendador
DETAIL_FIELDS = 'studios,rating,main_picture,related_anime,themes,demographics'

def fetch_detail(row):
    for intento in range(3):
        try:
            res = requests.get(
                f'https://api.myanimelist.net/v2/anime/{row["mal_id"]}',
                params={'fields': DETAIL_FIELDS},
                headers=headers,
                timeout=10
            )
            if res.status_code == 200:
                data = res.json()
                return {
                    'mal_id'       : row['mal_id'],
                    'studios'      : ', '.join([s['name'] for s in data.get('studios', [])]),
                    'rating'       : data.get('rating'),
                    'image_url'    : data.get('main_picture', {}).get('large'),
                    'related_anime': [r['node']['title'] for r in data.get('related_anime', [])],
                }
            elif res.status_code == 429:
                time.sleep(2 ** intento)
        except Exception as e:
            time.sleep(1)

    return {'mal_id': row['mal_id']}

detalles = []
rows = [row for _, row in airing.iterrows()]

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(fetch_detail, row): row for row in rows}
    for i, future in enumerate(as_completed(futures)):
        detalles.append(future.result())
        if (i + 1) % 50 == 0:
            print(f"   {i + 1}/{len(rows)} procesados...")

df_detalles = pd.DataFrame(detalles)
airing = airing.merge(df_detalles, on='mal_id', how='left')

print(f" airing enriquecido: {airing.shape}")
airing.head(3)

   50/349 procesados...
   100/349 procesados...
   150/349 procesados...
   200/349 procesados...
   250/349 procesados...
   300/349 procesados...
 airing enriquecido: (349, 15)


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis,studios,rating,image_url,related_anime
0,61469,1,Steel Ball Run: JoJo no Kimyou na Bouken,9.17,0,currently_airing,ona,"Action, Adventure, Historical, Mystery, Racing...",2026,1484,"In the American Old West, the world's greatest...",David Production,r,https://myanimelist.net/images/anime/1448/1541...,[JoJo no Kimyou na Bouken Part 6: Stone Ocean ...
1,51553,2,Tongari Boushi no Atelier,8.80,13,currently_airing,tv,"Fantasy, Seinen",2026,1824,"Coco, a humble dressmaker's daughter, has alwa...",BUG FILMS,pg_13,https://myanimelist.net/images/anime/1726/1555...,[]
2,21,3,One Piece,8.73,0,currently_airing,tv,"Action, Adventure, Fantasy, Shounen",1999,17,Barely surviving in a barrel after passing thr...,Toei Animation,pg_13,https://myanimelist.net/images/anime/1244/1388...,"[One Piece Movie 01, One Piece Movie 02: Nejim..."


In [18]:
## Extraccion de anime todo tipo para obtener animes ONA
mal_client_id = userdata.get('MyAnimeList')
headers = {'X-MAL-CLIENT-ID': mal_client_id}

todos = []
for offset in range(0, 3000, 100):
    res = requests.get('https://api.myanimelist.net/v2/anime/ranking', params={
        'ranking_type': 'all',
        'limit': 100,
        'offset': offset,
        'fields': 'rank,mean,genres,num_episodes,status,start_date,synopsis,popularity,media_type'
    }, headers=headers).json()

    for item in res.get('data', []):
        node = item['node']
        todos.append({
            'mal_id':     node['id'],
            'rank':       item['ranking']['rank'],
            'title':      node['title'],
            'score':      node.get('mean'),
            'episodes':   node.get('num_episodes'),
            'status':     node.get('status'),
            'media_type': node.get('media_type'),
            'genres':     ', '.join([g['name'] for g in node.get('genres', [])]),
            'year':       node.get('start_date', '')[:4],
            'popularity': node.get('popularity'),
            'synopsis':   node.get('synopsis'),
        })
    time.sleep(1)

ona = pd.DataFrame(todos)
print(f"{len(ona)} animes extraídos del ranking")
ona.head()

3000 animes extraídos del ranking


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis
0,52991,1,Sousou no Frieren,9.27,28,finished_airing,tv,"Adventure, Award Winning, Drama, Fantasy, Shounen",2023,104,During their decade-long quest to defeat the D...
1,61469,2,Steel Ball Run: JoJo no Kimyou na Bouken,9.17,0,currently_airing,ona,"Action, Adventure, Historical, Mystery, Racing...",2026,1484,"In the American Old West, the world's greatest..."
2,5114,3,Fullmetal Alchemist: Brotherhood,9.11,64,finished_airing,tv,"Action, Adventure, Drama, Fantasy, Military, S...",2009,3,After a horrific alchemy experiment goes wrong...
3,57555,4,Chainsaw Man Movie: Reze-hen,9.09,1,finished_airing,movie,"Action, Fantasy, Gore, Shounen, Urban Fantasy",2025,591,Despite the immediate challenges following bec...
4,9253,5,Steins;Gate,9.07,24,finished_airing,tv,"Drama, Psychological, Sci-Fi, Suspense, Time T...",2011,14,Eccentric scientist Rintarou Okabe has a never...


In [19]:
ona1=ona[ona["media_type"]=="ona"].copy()

In [20]:
ona1

,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis
1,61469,2,Steel Ball Run: JoJo no Kimyou na Bouken,9.17,0,currently_airing,ona,"Action, Adventure, Historical, Mystery, Racing...",2026,1484,"In the American Old West, the world's greatest..."
48,60489,49,Takopii no Genzai,8.76,6,finished_airing,ona,"Drama, Psychological, Sci-Fi, Shounen, Time Tr...",2025,914,"A squid-like creature, known as a Happian, lea..."
68,44074,69,Shiguang Dailiren,8.70,11,finished_airing,ona,"Adult Cast, Drama, Mystery, Super Power, Suspe...",2021,542,It is said that a picture is worth a thousand ...
74,53447,75,Tu Bian Yingxiong X,8.68,24,finished_airing,ona,"Action, Super Power",2025,1164,This is a world where heroes are created by pe...
83,57864,84,Monogatari Series: Off & Monster Season,8.65,14,finished_airing,ona,"Comedy, Mystery, Supernatural, Vampire",2024,2346,Koyomi Araragi spent his last year of high sch...
...,...,...,...,...,...,...,...,...,...,...,...
2958,38528,2959,Quanzhi Fashi III,7.33,12,finished_airing,ona,"Action, Fantasy, School",2018,3161,After the crisis brought forth by the Black Or...
2969,42284,2970,Yuan Long,7.33,16,finished_airing,ona,"Action, Adventure, Fantasy, Historical",2020,9469,Special Forces Wang Sheng traveled to the soul...
2974,49066,2975,Artiswitch,7.32,6,finished_airing,ona,"Drama, Fantasy, Music",2021,5345,There is a rumor doing the rounds in Harajuku:...
2996,36824,2997,Huyao Xiao Hongniang 5: Nan Guo Pian,7.32,26,finished_airing,ona,"Comedy, Historical, Romance, Supernatural",2017,7885,Fifth season of Huyao Xiao Hongniang.


In [21]:
ona1.reset_index(drop=True, inplace=True)

In [22]:
## Extracción de detalles para el funcionamiento del recomendador
DETAIL_FIELDS = 'studios,rating,main_picture,related_anime,themes,demographics'

def fetch_detail(row):
    for intento in range(3):
        try:
            res = requests.get(
                f'https://api.myanimelist.net/v2/anime/{row["mal_id"]}',
                params={'fields': DETAIL_FIELDS},
                headers=headers,
                timeout=10
            )
            if res.status_code == 200:
                data = res.json()
                return {
                    'mal_id'       : row['mal_id'],
                    'studios'      : ', '.join([s['name'] for s in data.get('studios', [])]),
                    'rating'       : data.get('rating'),
                    'image_url'    : data.get('main_picture', {}).get('large'),
                    'related_anime': [r['node']['title'] for r in data.get('related_anime', [])],
                }
            elif res.status_code == 429:
                time.sleep(2 ** intento)
        except Exception as e:
            time.sleep(1)

    return {'mal_id': row['mal_id']}

detalles = []
rows = [row for _, row in ona1.iterrows()]

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(fetch_detail, row): row for row in rows}
    for i, future in enumerate(as_completed(futures)):
        detalles.append(future.result())
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(rows)} procesados...")

df_detalles = pd.DataFrame(detalles)
ona1 = ona1.merge(df_detalles, on='mal_id', how='left')

print(f" ONA enriquecido: {ona1.shape}")
ona1.head(3)

  50/323 procesados...
  100/323 procesados...
  150/323 procesados...
  200/323 procesados...
  250/323 procesados...
  300/323 procesados...
 ONA enriquecido: (323, 15)


,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis,studios,rating,image_url,related_anime
0,61469,2,Steel Ball Run: JoJo no Kimyou na Bouken,9.17,0,currently_airing,ona,"Action, Adventure, Historical, Mystery, Racing...",2026,1484,"In the American Old West, the world's greatest...",David Production,r,https://myanimelist.net/images/anime/1448/1541...,[JoJo no Kimyou na Bouken Part 6: Stone Ocean ...
1,60489,49,Takopii no Genzai,8.76,6,finished_airing,ona,"Drama, Psychological, Sci-Fi, Shounen, Time Tr...",2025,914,"A squid-like creature, known as a Happian, lea...",Enishiya,r,https://myanimelist.net/images/anime/1182/1498...,[]
2,44074,69,Shiguang Dailiren,8.70,11,finished_airing,ona,"Adult Cast, Drama, Mystery, Super Power, Suspe...",2021,542,It is said that a picture is worth a thousand ...,LAN Studio,pg_13,https://myanimelist.net/images/anime/1135/1148...,"[Shiguang Dailiren Fanwai Pian: Biwu Zhaoqin, ..."


In [23]:
dfs = [tv, ova, special, airing, ona1, movie, favorite]

# Unir todos
df_anime = pd.concat(dfs, ignore_index=True)

In [24]:
df_anime

,mal_id,rank,title,score,episodes,status,media_type,genres,year,popularity,synopsis,studios,rating,image_url,related_anime
0,52991,1,Sousou no Frieren,9.27,28,finished_airing,tv,"Adventure, Award Winning, Drama, Fantasy, Shounen",2023,104,During their decade-long quest to defeat the D...,Madhouse,pg_13,https://myanimelist.net/images/anime/1015/1380...,"[Yuusha, Sousou no Frieren: ●● no Mahou, Haru ..."
1,5114,2,Fullmetal Alchemist: Brotherhood,9.11,64,finished_airing,tv,"Action, Adventure, Drama, Fantasy, Military, S...",2009,3,After a horrific alchemy experiment goes wrong...,Bones,r,https://myanimelist.net/images/anime/1208/9474...,"[Fullmetal Alchemist, Fullmetal Alchemist: Bro..."
2,9253,3,Steins;Gate,9.07,24,finished_airing,tv,"Drama, Psychological, Sci-Fi, Suspense, Time T...",2011,14,Eccentric scientist Rintarou Okabe has a never...,White Fox,pg_13,https://myanimelist.net/images/anime/1935/1279...,"[ChäoS;HEAd, Steins;Gate: Oukoubakko no Poriom..."
3,28977,4,Gintama°,9.05,51,finished_airing,tv,"Action, Comedy, Gag Humor, Historical, Parody,...",2015,349,"Gintoki, Shinpachi, and Kagura return as the f...",Bandai Namco Pictures,pg_13,https://myanimelist.net/images/anime/3/72078l.jpg,"[Gintama': Enchousen, Gintama°: Umai-mono wa A..."
4,38524,5,Shingeki no Kyojin Season 3 Part 2,9.05,10,finished_airing,tv,"Action, Drama, Gore, Military, Shounen, Surviv...",2019,20,Seeking to restore humanity's diminishing hope...,Wit Studio,r,https://myanimelist.net/images/anime/1517/1006...,"[Shingeki no Kyojin Season 3, Shingeki no Kyoj..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3167,54112,496,Zom 100: Zombie ni Naru made ni Shitai 100 no ...,7.70,12,finished_airing,tv,"Adult Cast, Comedy, Seinen, Survival, Suspense",2023,357,After graduating from a top university with an...,BUG FILMS,r,https://myanimelist.net/images/anime/1384/1364...,[Zom 100: Zombie ni Naru made ni Shitai 100 no...
3168,5040,497,One Outs,8.32,25,finished_airing,tv,"Adult Cast, Psychological, Seinen, Sports, Sus...",2008,1089,Toua Tokuchi is a prodigy when it comes to bot...,Madhouse,pg_13,https://myanimelist.net/images/anime/7/21065l....,[]
3169,16,498,Hachimitsu to Clover,7.98,24,finished_airing,tv,"Adult Cast, Comedy, Drama, Josei, Love Polygon...",2005,983,"Yuuta Takemoto, a sophomore at an arts college...",J.C.Staff,pg_13,https://myanimelist.net/images/anime/1301/1335...,"[Hachimitsu to Clover Specials, Hachimitsu to ..."
3170,5681,499,Summer Wars,8.01,1,finished_airing,movie,"Award Winning, Comedy, Sci-Fi",2009,457,"OZ, a virtual world connected to the internet,...",Madhouse,pg_13,https://myanimelist.net/images/anime/1593/1167...,[]


In [25]:
#df_anime.to_csv('anime.csv', index=False, encoding='utf-8')

#### TMDB

In [26]:
from google.colab import userdata
tmdb_api_key = userdata.get('TMDB')

In [27]:
## Extraccion de peliculas y series

tmdb_api_key = userdata.get('TMDB')
base_url = 'https://api.themoviedb.org/3'

#  Mapeo de géneros
def get_tmdb_genre_map():
    genre_map = {}
    for media in ['movie', 'tv']:
        res = requests.get(f'{base_url}/genre/{media}/list',
                           params={'api_key': tmdb_api_key, 'language': 'es-MX'}).json()
        for g in res['genres']:
            genre_map[g['id']] = g['name']
    return genre_map

tmdb_genre_map = get_tmdb_genre_map()

#  Público
def get_certification(tmdb_id, tipo):
    """
    Obtiene la clasificación por edades (G, PG, R, TV-MA, etc.)
    """
    sub_endpoint = 'release_dates' if tipo == 'película' else 'content_ratings'
    media_type = 'movie' if tipo == 'película' else 'tv'

    try:
        res = requests.get(f'{base_url}/{media_type}/{tmdb_id}/{sub_endpoint}',
                           params={'api_key': tmdb_api_key}).json()

        results = res.get('results', [])

        for r in results:
            if r['iso_3166_1'] in ['US', 'MX']:
                if tipo == 'película':

                    return r['release_dates'][0].get('certification', 'N/A')
                else:

                    return r.get('rating', 'N/A')
        return 'N/A'
    except:
        return 'N/A'

#  Idiomas
IDIOMAS = ['en', 'es', 'fr', 'ko', 'it', 'de', 'zh', 'pt']

#  PELÍCULAS
def get_top_movies(paginas=100):
    rows = []
    for idioma in IDIOMAS:
        print(f"   Procesando Películas [{idioma}]...")
        for page in range(1, paginas + 1):
            try:
                res = requests.get(f'{base_url}/discover/movie', params={
                    'api_key'                : tmdb_api_key,
                    'language'               : 'es-MX',
                    'with_original_language' : idioma,
                    'sort_by'                : 'vote_average.desc',
                    'vote_count.gte'         : 1000,
                    'page'                   : page
                }).json()

                for m in res.get('results', []):
                    cert = get_certification(m['id'], 'película')

                    rows.append({
                        'tmdb_id'        : m['id'],
                        'tipo'           : 'película',
                        'título'         : m['title'],
                        'público'        : cert,
                        'score'          : m['vote_average'],
                        'votos'          : m['vote_count'],
                        'fecha'          : m['release_date'],
                        'géneros'        : [tmdb_genre_map.get(g, 'Unknown') for g in m['genre_ids']],
                        'sinopsis'       : m.get('overview', ''),
                        'popularidad'    : m['popularity'],
                        'idioma_original': idioma,
                        'imagen_url'     : f"https://image.tmdb.org/t/p/w500{m['poster_path']}" if m.get('poster_path') else None,
                    })
                time.sleep(0.2)
            except Exception as e:
                print(f"     Error página {page} [{idioma}]: {e}")
    return rows

#  SERIES
def get_top_series(paginas=100):
    rows = []
    for idioma in IDIOMAS:
        print(f"  Procesando Series [{idioma}]...")
        for page in range(1, paginas + 1):
            try:
                res = requests.get(f'{base_url}/discover/tv', params={
                    'api_key'                : tmdb_api_key,
                    'language'               : 'es-MX',
                    'with_original_language' : idioma,
                    'sort_by'                : 'vote_average.desc',
                    'vote_count.gte'         : 500,
                    'page'                   : page
                }).json()

                for s in res.get('results', []):
                    cert = get_certification(s['id'], 'serie')

                    rows.append({
                        'tmdb_id'        : s['id'],
                        'tipo'           : 'serie',
                        'título'         : s['name'],
                        'público'        : cert,
                        'score'          : s['vote_average'],
                        'votos'          : s['vote_count'],
                        'fecha'          : s.get('first_air_date', 'N/A'),
                        'géneros'        : [tmdb_genre_map.get(g, 'Unknown') for g in s['genre_ids']],
                        'sinopsis'       : s.get('overview', ''),
                        'popularidad'    : s['popularity'],
                        'idioma_original': idioma,
                        'imagen_url'     : f"https://image.tmdb.org/t/p/w500{s['poster_path']}" if s.get('poster_path') else None,
                    })
                time.sleep(0.2)
            except Exception as e:
                print(f"  Error página {page} [{idioma}]: {e}")
    return rows

#  EJECUCIÓN PRINCIPAL
print(" Iniciando extracción...")
peliculas_data = get_top_movies(paginas=100)
series_data = get_top_series(paginas=100)

df_pelser = pd.DataFrame(peliculas_data + series_data)
df_pelser = df_pelser.drop_duplicates(subset='tmdb_id').reset_index(drop=True)
df_pelser = df_pelser.sort_values('score', ascending=False).reset_index(drop=True)

print(f"\nDataset final: {df_pelser.shape}")
print(df_pelser[['título', 'tipo', 'público', 'score']].head(10))

 Iniciando extracción...
   Procesando Películas [en]...
   Procesando Películas [es]...
   Procesando Películas [fr]...
   Procesando Películas [ko]...
   Procesando Películas [it]...
   Procesando Películas [de]...
   Procesando Películas [zh]...
   Procesando Películas [pt]...
  Procesando Series [en]...
  Procesando Series [es]...
  Procesando Series [fr]...
  Procesando Series [ko]...
  Procesando Series [it]...
  Procesando Series [de]...
  Procesando Series [zh]...
  Procesando Series [pt]...

Dataset final: (3221, 12)
                       título      tipo público  score
0                Breaking Bad     serie   TV-MA   8.94
1  Avatar: La leyenda de Aang     serie   TV-Y7   8.80
2                      Arcane     serie   TV-14   8.76
3               Sueño de fuga  película       B   8.72
4            Better Call Saul     serie   TV-MA   8.71
5                 Los Soprano     serie   TV-MA   8.70
6                   Chernobyl     serie   TV-MA   8.70
7                    The Pit

In [28]:
df_pelser

,tmdb_id,tipo,título,público,score,votos,fecha,géneros,sinopsis,popularidad,idioma_original,imagen_url
0,1396,serie,Breaking Bad,TV-MA,8.94,17455,2008-01-20,"[Drama, Crimen]",Un profesor de Química de secundaria con cánce...,130.68,en,https://image.tmdb.org/t/p/w500/ztkUQFLlC19CCM...
1,246,serie,Avatar: La leyenda de Aang,TV-Y7,8.80,4753,2005-02-21,"[Animación, Action & Adventure, Sci-Fi & Fantasy]","El mundo está dividido en cuatro naciones, cor...",22.21,en,https://image.tmdb.org/t/p/w500/ucNtkZfpZ6Kgxq...
2,94605,serie,Arcane,TV-14,8.76,5793,2021-11-06,"[Animación, Action & Adventure, Sci-Fi & Fantasy]",Mientras la discordia separa las ciudades geme...,39.35,en,https://image.tmdb.org/t/p/w500/i1vSPUbiBe2iK2...
3,278,película,Sueño de fuga,B,8.72,30063,1994-09-23,"[Drama, Crimen]",Andy Dufresne es un banquero injustamente enca...,55.62,en,https://image.tmdb.org/t/p/w500/l6V3TEoJlGh3zt...
4,60059,serie,Better Call Saul,TV-MA,8.71,6333,2015-02-08,"[Crimen, Drama]",La acción se ubica en el año 2002 en torno al ...,103.76,en,https://image.tmdb.org/t/p/w500/zjg4jpK1Wp2kiR...
...,...,...,...,...,...,...,...,...,...,...,...,...
3216,80280,película,[REC] 3: El comienzo,C,5.32,1602,2012-03-30,[Terror],Koldo y Clara están hechos el uno para el otro...,3.18,es,https://image.tmdb.org/t/p/w500/cRVSZ03HIXKOKf...
3217,2395,película,Astérix aux Jeux olympiques,N/A,5.20,2086,2008-01-25,"[Fantasía, Aventura, Comedia, Familia]",,3.25,fr,https://image.tmdb.org/t/p/w500/gSA4pkT2h2bgrb...
3218,99770,película,Astérix y Obélix al servicio de su majestad,A,5.11,1390,2012-10-17,"[Familia, Aventura, Comedia]",,3.17,fr,https://image.tmdb.org/t/p/w500/vGbzyJFyZtOeyU...
3219,130392,serie,The D'Amelio Show,TV-14,5.00,684,2021-09-03,[Reality],,3.97,en,https://image.tmdb.org/t/p/w500/1TMOWefVY5854o...


In [29]:
#df_pelser.to_csv('pelser.csv', index=False, encoding='utf-8')